# 06 -- DreamBooth LoRA: teaching Stable Diffusion who Ray is

Notebooks 01-05 trained models **from scratch** on 64x64 images. With 258 photos that is enough to learn
*roughly* what Ray looks like, and not much more. This notebook takes the other route: start from
**Stable Diffusion 1.5** -- trained on billions of image-text pairs, so it already knows dogs, beaches,
oil paintings and lighting -- and teach it one new concept: *this particular dog*.

After ~2000 steps (30-45 min on a 24 GB GPU) the prompt `a photo of sks dog on the beach at sunset`
produces Ray, at 512x512, in a scene that is in none of the photos.

**In this notebook:** latent diffusion recap, DreamBooth (identifier tokens, prior preservation, language drift),
LoRA, training on Ray_dog, the LoRA strength dial, checking for memorisation and drift, and pushing the LoRA
to the Hub.

*Papers:* Rombach et al., *Latent Diffusion* (2022) · Ruiz et al., *DreamBooth* (2022) · Hu et al., *LoRA* (2021).

## 1. Theory

### Stable Diffusion = the diffusion model of notebook 04, in a latent space, with text
* A frozen **VAE** (a much better one than notebook 01's: trained with perceptual and adversarial losses) compresses a
  $512\times512\times3$ image to a $64\times64\times4$ latent $z$. Diffusion runs on $z$ -- 48x fewer values.
* A frozen **CLIP text encoder** turns the prompt into 77 token embeddings $c$.
* The **U-Net** $\varepsilon_\theta(z_t, t, c)$ predicts the noise, reading the prompt through **cross-attention**:
  queries come from image features, keys and values from the text tokens, $\operatorname{softmax}(QK^\top/\sqrt d)V$.
  This is where "which words affect which pixels" is decided.
* Training loss: exactly notebook 04's, $\mathbb E\,\|\varepsilon-\varepsilon_\theta(z_t,t,c)\|^2$. Sampling uses classifier-free guidance
  (scale 7.5) and a fast solver (DPM-Solver++, 25 steps).

### DreamBooth
Fine-tune on the subject's photos, every one captioned with the same prompt: **`a photo of sks dog`**.
* **`sks`** is a rare token: it has almost no meaning to the model, so the gradient has nowhere to put "Ray's identity"
  except into it. (Using `Ray` would collide with what the model knows about the word.)
* **`dog`** (the *class noun*) lets the model reuse everything it knows about dogs; only the difference has to be learnt.
* **Language drift:** after fine-tuning, *every* dog starts to look like Ray, because "dog" was in every caption.
  **Prior preservation** fixes this: before training, generate ~200 images of `a photo of a dog` with the *original*
  model, and train on both, $\mathcal L=\mathcal L_\text{instance}+\lambda\,\mathcal L_\text{prior}$. The model is told "sks dog -> Ray, but dog -> any dog".
* **Overfitting:** too many steps and outputs become near-copies of training photos (same pose, same background), and
  the prompt stops mattering. Watch the validation grid; stop when new scenes still render.

### LoRA
Full fine-tuning updates all 860M U-Net weights: a 3.4 GB file per subject, and easy to overfit. LoRA freezes $W$ and learns
a low-rank update
$$
W' = W + \frac{\alpha}{r}\,BA,\qquad B\in\mathbb R^{d\times r},\ A\in\mathbb R^{r\times k},\ r\ll d,k
$$
on the attention projections ($W_q, W_k, W_v, W_o$). $B$ starts at zero, so training starts *exactly* at the base model.
Rank 16 -> ~3M parameters -> a **12 MB** file that loads on top of any copy of SD 1.5, and whose **strength can be dialled**
at inference ($W+s\frac{\alpha}{r}BA$, $s\in[0,1]$). Multiple LoRAs can be combined.

### What trains here
Only the U-Net LoRA ($\approx$0.4% of the U-Net). VAE, text encoder and base U-Net weights are frozen in bf16; the LoRA
weights train in fp32. A checkpoint is the LoRA + optimizer state (~40 MB), not the 4 GB model.

## 0. Setup

Run from the repo root or from `notebooks/`. The package is imported from the repo,
so editing `genmodels/*.py` and re-running a cell picks up the change after a kernel restart.

On **RunPod**: `bash scripts/runpod/setup.sh` once, then open JupyterLab (port 8888).
On **Colab**: uncomment the first line.

In [ ]:
# !git clone -q https://github.com/yar78/vision-generative-models-survey.git && %cd vision-generative-models-survey && pip install -q -e .
import sys, pathlib
REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import torch
import yaml
from genmodels import build_trainer
from genmodels.common.config import load_config
from genmodels.common.data import get_dataset
from genmodels.common.utils import get_device, load_images, nearest_neighbours, plot_history, show_grid
from genmodels.hub import push_run, resolve_repo_id
from genmodels.dreambooth_lora import Trainer as DreamBoothLoRATrainer
import torch.nn.functional as F
print("device:", get_device())

**This notebook needs a GPU** (24 GB recommended; set `gradient_checkpointing: true` for 12-16 GB). On CPU, building the model and sampling work but take minutes per image.

In [ ]:
if not torch.cuda.is_available():
    print("WARNING: no CUDA GPU -- training will be impractically slow. Use a RunPod GPU pod.")
else:
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")

## 2. The experiment

`configs/ray_dog/dreambooth_lora.yaml`: Ray_dog at 512x512 (random crops covering 70-100% of each photo, flips),
SD 1.5, LoRA rank 16, prior preservation with 200 class images, 2000 steps at batch 4, lr 1e-4.
The validation prompts (sampled every 250 steps) put `sks dog` in new scenes and styles, and include
`a photo of a dog` to watch for language drift.

In [ ]:
CONFIG = REPO / "configs/ray_dog/dreambooth_lora.yaml"
cfg = load_config(CONFIG)

# Override anything here, for example:
# cfg["train"]["steps"] = 1000
# cfg["method"]["with_prior_preservation"] = False
# cfg["method"]["instance_prompt"] = "a photo of ohwx dog"   # then update validation_prompts too

print(yaml.safe_dump(cfg, sort_keys=False))

### The training photos, as the model will see them (512x512 crops)

In [ ]:
train_set = get_dataset(dict(cfg["data"], num_workers=0))
print(f"{len(train_set)} training images")
x_real = torch.stack([train_set[i][0] for i in torch.randperm(len(train_set))[:8].tolist()])
show_grid(x_real, nrow=4, title="training crops", size=10)

## 3. Load Stable Diffusion and add LoRA

Downloads SD 1.5 the first time (~2 GB in fp16; cached under `HF_HOME`, which on RunPod is on the network volume).

In [ ]:
trainer = build_trainer(cfg)
print("run dir:", trainer.run_dir)
print({k: f"{v/1e6:.1f}M" for k, v in trainer.num_params().items()})
lora_layers = [n for n, _ in trainer.unet.named_modules() if n.endswith("lora_A.default")]
print(len(lora_layers), "LoRA-adapted layers, e.g.", lora_layers[:3])

### Before fine-tuning

The LoRA starts as an exact no-op ($B=0$), so these are the **base model's** answers. `sks dog` is just "some dog"
(sometimes with odd artefacts -- the token is nearly meaningless); this is the baseline to compare against after training.

In [ ]:
before = trainer.sample(8, seed=0)
show_grid(before, nrow=4, title="base SD 1.5 on the validation prompts", size=12)
for i, p in enumerate(trainer.validation_prompts):
    print(i, p)

### The training objective, as implemented
Notebook 04's loss, on VAE latents and with the prompt embedding.

In [ ]:
import inspect; print(inspect.getsource(DreamBoothLoRATrainer.diffusion_loss))

## 4. Train

With prior preservation on, the first thing `fit()` does is generate the class images (`a photo of a dog`, 200 images,
~3 min on a 4090) into `<run dir>/class_images/`; they are reused on resume. Then 2000 LoRA steps.

`loss_instance` and `loss_prior` are noisy and barely move -- that is normal for diffusion (it is an average over random
noise levels). **Judge progress by the sample grids** in TensorBoard (every 250 steps).

For long runs prefer the script, which keeps going if the browser disconnects:
`bash scripts/runpod/train.sh configs/ray_dog/dreambooth_lora.yaml --push-to-hub` -- then set `TRAIN = False` here to load its result.

In [ ]:
TRAIN = True

if TRAIN:
    trainer.fit()
elif trainer.ckpt_path.exists():
    trainer.load_checkpoint()
    print(f"loaded LoRA checkpoint at step {trainer.step}")
else:
    print("No checkpoint yet in", trainer.run_dir)

### TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {trainer.run_dir}/tensorboard

In [ ]:
plot_history(trainer.history)

## 5. Results
### Before vs after, same seeds and prompts

In [ ]:
after = trainer.sample(8, seed=0)
show_grid(torch.cat([before, after]), nrow=8, title="top: base SD 1.5   bottom: + Ray LoRA", size=16)

### Training progress
The saved validation grids. Early: generic dogs. Middle: Ray in new scenes. Too late: the scenes start to turn into the training photos.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
grids = sorted((trainer.run_dir / "samples").glob("step_*.png"))
for p in grids[:: max(1, len(grids) // 4)][-4:]:
    plt.figure(figsize=(16, 8)); plt.imshow(Image.open(p)); plt.title(p.stem); plt.axis("off"); plt.show()

### Your own prompts
Keep `sks dog` in the prompt. Styles, places, seasons, art media all work.

In [ ]:
prompts = [
    "a photo of sks dog as an astronaut on the moon",
    "a children's book illustration of sks dog",
    "a photo of sks dog sleeping on a sofa, cozy lighting",
    "a pixar style 3d render of sks dog",
]
show_grid(trainer.sample(len(prompts), prompts=prompts, seed=1), nrow=4, size=16)

## 6. Inspecting what was learnt

### The LoRA strength dial
Same prompt and seed, LoRA scale $s = 0, 0.25, 0.5, 0.75, 1$. Identity fades in; if $s=1$ copies the photos too closely,
a lower scale often keeps the identity and recovers prompt fidelity.

In [ ]:
prompt = ["an oil painting of sks dog in the style of van gogh"]
row = torch.cat([trainer.sample(1, prompts=prompt, seed=3, lora_scale=s) for s in (0.0, 0.25, 0.5, 0.75, 1.0)])
show_grid(row, nrow=5, title="LoRA scale 0 / 0.25 / 0.5 / 0.75 / 1", size=16)

### Language drift: does every dog look like Ray now?
`a photo of a dog` without and with the LoRA. With prior preservation the bottom row should still show many
breeds; with `with_prior_preservation: false` it usually turns into Ray.

In [ ]:
dog = ["a photo of a dog"] * 4
show_grid(torch.cat([trainer.sample(4, prompts=dog, seed=5, lora_scale=0.0), trainer.sample(4, prompts=dog, seed=5)]),
          nrow=4, title="'a photo of a dog' -- top: base   bottom: with LoRA", size=12)

### Memorisation check
Generated `a photo of sks dog` images (top) and their nearest training photo (bottom), compared at 128 px.
Some resemblance is the point (it is Ray); an identical pose *and* background means the LoRA is replaying photos --
fewer steps, a lower rank or a lower LoRA scale help.

In [ ]:
eval_set = get_dataset(dict(cfg["data"], image_size=128, cache_size=144, random_crop=False, hflip=False, num_workers=0))
train_imgs = load_images(eval_set)
samples = trainer.sample(8, prompts=[trainer.instance_prompt], seed=7)
small = F.interpolate(samples, size=128, mode="bilinear", antialias=True)
show_grid(torch.cat([small, nearest_neighbours(small, train_imgs)[:, 0]]), nrow=8,
          title="samples (top) vs nearest training photo (bottom)", size=16)

## 7. Push the LoRA to the Hugging Face Hub

`push_run` uploads a **diffusers LoRA repo** (private by default) named `<your-username>/{cfg['name']}`:
`pytorch_lora_weights.safetensors` (~12 MB), a model card with `base_model` and `instance_prompt` metadata (so the Hub
shows it as a LoRA of SD 1.5 and the inference widget knows the trigger words), the sample grid, metrics, config and
TensorBoard logs. SD 1.5's CreativeML OpenRAIL-M license applies to the LoRA.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, whoami
    try:
        print("logged in as", whoami()["name"])
    except Exception:
        notebook_login()
    url = push_run(trainer.run_dir)

### Use it anywhere
The standard diffusers way -- this is all anyone needs to generate Ray:

In [ ]:
if PUSH_TO_HUB:
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(cfg["model"]["pretrained"], torch_dtype=torch.float16).to("cuda")
    pipe.load_lora_weights(resolve_repo_id(cfg))
    image = pipe("a photo of sks dog on a sailboat", num_inference_steps=30, guidance_scale=7.5).images[0]
    display(image)

## 8. Things to try

* **Prior preservation off** (`with_prior_preservation: false`): faster, but check the drift cell.
* **Rank** 4 vs 16 vs 64: identity fidelity vs overfitting vs file size.
* **Steps**: compare the 500 / 1000 / 2000 grids; 258 photos is far more than DreamBooth's usual 5-20, so fewer steps may suffice.
* **Fewer, cleaner photos**: hand-pick 20-30 where Ray is large and alone. Some photos include people, who can leak into samples.
* **Captions**: per-photo captions ("sks dog lying on grass") let the model separate Ray from backgrounds.
* **Bigger base models**: SDXL (1024 px) or a flow-matching model (SD3 / FLUX) -- notebook 05's objective at scale.